# Spectral scale selection with Rényi divergence

This first experiment:

1. samples $n$ observations from $N(0,1)$;
2. constructs a user-controlled geometric bandwidth grid;
3. evaluates the Gaussian KDE for every bandwidth;
4. computes the Rényi divergence matrix

$$
D_\alpha(p_i \Vert p_j)
= \frac{1}{\alpha-1}
  \log \int p_i(x)^\alpha p_j(x)^{1-\alpha} \, \mathrm{d}x.
$$

5. assembles $S_{ij}=\exp\{-\tau D_\alpha(p_i \Vert p_j)\}$; and
6. plots the stationary distribution of the Doob $h$-transform and its Toeplitz-corrected score.

For $\alpha=1/2$, Rényi divergence is symmetric:

$$
D_{1/2}(p_i \Vert p_j)
= -2 \log \int \sqrt{p_i(x)p_j(x)} \, \mathrm{d}x.
$$

In [7]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from kde_tools.fft import kde_family_fft
from kde_tools.similarities import (
    doob_stationary_mass,
    renyi_divergence_matrix,
    toeplitz_corrected_choice,
    toeplitz_projection,
)

np.set_printoptions(precision=5, suppress=True)

## 1. Experiment parameters

The bandwidth grid is fixed before the sample is generated. Change `h_min`, `h_max`, and `N` to control its range and resolution. The spatial parameters control only the numerical KDE evaluation, not the bandwidth candidates.

In [19]:
seed = 20260818
n = 500

h_min = 0.02
h_max = 2.00
N = 100

alpha = 0.5
tau = 1.0

half_width = 12.0
spatial_points = 8192

if n < 1:
    raise ValueError("n must be positive.")
if not 0.0 < h_min < h_max:
    raise ValueError("Require 0 < h_min < h_max.")
if N < 2:
    raise ValueError("N must be at least 2.")
if alpha <= 0.0:
    raise ValueError("alpha must be positive.")
if tau <= 0.0:
    raise ValueError("tau must be positive.")

h_grid = np.geomspace(h_min, h_max, N)
rng = np.random.default_rng(seed)
samples = rng.normal(loc=0.0, scale=1.0, size=n)

print(f"sample size: {n}")
print(
    f"bandwidth grid: [{h_grid[0]:.5g}, {h_grid[-1]:.5g}] "
    f"with {len(h_grid)} points"
)

sample size: 500
bandwidth grid: [0.02, 2] with 100 points


## 2. KDE family, Rényi divergence, and similarity matrix

The KDE family is evaluated on a fine periodic grid by the reusable FFT routine. The endpoint is appended only for trapezoidal integration. At $\alpha=1/2$, $D_\alpha$ should be symmetric; the explicit averaging below removes negligible floating-point asymmetry.

In [20]:
density_values, dx = kde_family_fft(
    samples=samples,
    bandwidths=h_grid,
    half_width=half_width,
    spatial_points=spatial_points,
)

x_grid = np.arange(spatial_points + 1, dtype=float) * dx
periodic_density_values = np.concatenate(
    (density_values, density_values[:, :1]),
    axis=1,
)

D_alpha = renyi_divergence_matrix(
    density_values=periodic_density_values,
    x_grid=x_grid,
    alpha=alpha,
    bandwidths=h_grid,
)

if np.isclose(alpha, 0.5):
    D_alpha = 0.5 * (D_alpha + D_alpha.T)

similarity_matrix = np.exp(-tau * D_alpha)
np.fill_diagonal(similarity_matrix, 1.0)

print(
    "maximum symmetry error in D_alpha: "
    f"{np.max(np.abs(D_alpha - D_alpha.T)):.3e}"
)
print(
    f"similarity range: "
    f"[{similarity_matrix.min():.5f}, {similarity_matrix.max():.5f}]"
)

maximum symmetry error in D_alpha: 0.000e+00
similarity range: [0.66956, 1.00000]


## 3. Doob stationary distribution and Toeplitz correction

If $Sr=\rho r$ and $\ell^\mathsf{T}S=\rho\ell^\mathsf{T}$, the stationary distribution of the Perron–Doob chain is

$$
\pi_i
= \frac{\ell_i r_i}{\sum_j \ell_j r_j}.
$$

For the symmetric $\alpha=1/2$ matrix, $\ell=r$, and hence $\pi_i \propto r_i^2$. The Toeplitz reference averages entries along equal bandwidth-index lags. The existing correction compares the Perron weights through $c_i=r_i/r_i^{(0)}$. Maximizing $c_i$ is equivalent to maximizing the stationary-mass ratio because the latter is proportional to $c_i^2$.

In [21]:
stationary_distribution = doob_stationary_mass(similarity_matrix)
stationary_index = int(np.argmax(stationary_distribution))
stationary_h = float(h_grid[stationary_index])

toeplitz_reference = toeplitz_projection(
    matrix=similarity_matrix,
    symmetric=True,
)
toeplitz_stationary_distribution = doob_stationary_mass(toeplitz_reference)

corrected_h, corrected_index, toeplitz_corrected_score = (
    toeplitz_corrected_choice(
        similarity=similarity_matrix,
        bandwidths=h_grid,
        directed=False,
    )
)

print(
    f"Doob stationary maximum: h = {stationary_h:.6g} "
    f"(index {stationary_index})"
)
print(
    f"Toeplitz-corrected maximum: h = {corrected_h:.6g} "
    f"(index {corrected_index})"
)

Doob stationary maximum: h = 0.430887 (index 66)
Toeplitz-corrected maximum: h = 0.02 (index 0)


In [22]:
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=(
        f"S = exp(-τ D_{alpha:g})",
        "Stationary Doob distribution",
        "Toeplitz-corrected spectral score",
    ),
    horizontal_spacing=0.08,
)

fig.add_trace(
    go.Heatmap(
        z=similarity_matrix,
        colorscale="Viridis",
        colorbar=dict(title="similarity", x=0.29),
        name="similarity",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=h_grid,
        y=stationary_distribution,
        mode="lines",
        line=dict(color="#1f77b4", width=3),
        name="Doob stationary distribution π",
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(
        x=h_grid,
        y=toeplitz_stationary_distribution,
        mode="lines",
        line=dict(color="#777777", width=2, dash="dash"),
        name="Toeplitz reference π⁽⁰⁾",
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(
        x=[stationary_h],
        y=[stationary_distribution[stationary_index]],
        mode="markers",
        marker=dict(color="#1f77b4", size=10),
        name=f"stationary maximum: h={stationary_h:.4g}",
    ),
    row=1,
    col=2,
)
fig.add_vline(
    x=stationary_h,
    line=dict(color="#1f77b4", dash="dot"),
    row=1,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=h_grid,
        y=toeplitz_corrected_score,
        mode="lines",
        line=dict(color="#ff7f0e", width=3),
        name="corrected score r/r⁽⁰⁾",
    ),
    row=1,
    col=3,
)
fig.add_trace(
    go.Scatter(
        x=[corrected_h],
        y=[toeplitz_corrected_score[corrected_index]],
        mode="markers",
        marker=dict(color="#ff7f0e", size=10),
        name=f"corrected maximum: h={corrected_h:.4g}",
    ),
    row=1,
    col=3,
)
fig.add_hline(
    y=1.0,
    line=dict(color="#777777", dash="dash"),
    row=1,
    col=3,
)
fig.add_vline(
    x=corrected_h,
    line=dict(color="#ff7f0e", dash="dot"),
    row=1,
    col=3,
)

fig.update_xaxes(title_text="bandwidth index j", row=1, col=1)
fig.update_yaxes(title_text="bandwidth index i", row=1, col=1)
fig.update_xaxes(title_text="bandwidth h", type="log", row=1, col=2)
fig.update_yaxes(title_text="stationary mass", row=1, col=2)
fig.update_xaxes(title_text="bandwidth h", type="log", row=1, col=3)
fig.update_yaxes(title_text="relative Perron weight", row=1, col=3)

fig.update_layout(
    title=f"N(0,1) sample: n={n}, α={alpha:g}, τ={tau:g}",
    width=1500,
    height=520,
    template="plotly_white",
    legend=dict(orientation="h", y=-0.22),
    margin=dict(t=90, b=130),
)
fig.show()

## 4. Sampling distribution of the selected bandwidth

Repeat the complete Toeplitz-corrected selection for $M=100$ independent samples from $N(0,1)$. Every repetition uses the same fixed geometric bandwidth grid. Because the selector returns a grid point, counting selections at each grid point gives the exact empirical histogram.

In [23]:
M = 200
monte_carlo_seed = seed + 1
monte_carlo_rng = np.random.default_rng(monte_carlo_seed)

selected_indices = np.empty(M, dtype=int)
selected_bandwidths = np.empty(M, dtype=float)

for repetition in range(M):
    repetition_samples = monte_carlo_rng.normal(
        loc=0.0,
        scale=1.0,
        size=n,
    )
    repetition_densities, repetition_dx = kde_family_fft(
        samples=repetition_samples,
        bandwidths=h_grid,
        half_width=half_width,
        spatial_points=spatial_points,
    )
    repetition_x_grid = (
        np.arange(spatial_points + 1, dtype=float) * repetition_dx
    )
    repetition_periodic_densities = np.concatenate(
        (repetition_densities, repetition_densities[:, :1]),
        axis=1,
    )
    repetition_divergence = renyi_divergence_matrix(
        density_values=repetition_periodic_densities,
        x_grid=repetition_x_grid,
        alpha=alpha,
        bandwidths=h_grid,
    )
    if np.isclose(alpha, 0.5):
        repetition_divergence = 0.5 * (
            repetition_divergence + repetition_divergence.T
        )

    repetition_similarity = np.exp(-tau * repetition_divergence)
    np.fill_diagonal(repetition_similarity, 1.0)
    selected_h, selected_index, _ = toeplitz_corrected_choice(
        similarity=repetition_similarity,
        bandwidths=h_grid,
        directed=False,
    )
    selected_indices[repetition] = selected_index
    selected_bandwidths[repetition] = selected_h

    if (repetition + 1) % 10 == 0 or repetition + 1 == M:
        print(f"completed {repetition + 1}/{M} repetitions")

selection_counts = np.bincount(selected_indices, minlength=N)
selected_grid_mask = selection_counts > 0

print(f"mean selected bandwidth: {selected_bandwidths.mean():.6g}")
print(f"median selected bandwidth: {np.median(selected_bandwidths):.6g}")
print(
    f"most frequent bandwidth: "
    f"{h_grid[int(np.argmax(selection_counts))]:.6g}"
)

completed 10/200 repetitions
completed 20/200 repetitions
completed 30/200 repetitions
completed 40/200 repetitions
completed 50/200 repetitions
completed 60/200 repetitions
completed 70/200 repetitions
completed 80/200 repetitions
completed 90/200 repetitions
completed 100/200 repetitions
completed 110/200 repetitions
completed 120/200 repetitions
completed 130/200 repetitions
completed 140/200 repetitions
completed 150/200 repetitions
completed 160/200 repetitions
completed 170/200 repetitions
completed 180/200 repetitions
completed 190/200 repetitions
completed 200/200 repetitions
mean selected bandwidth: 0.02
median selected bandwidth: 0.02
most frequent bandwidth: 0.02


In [24]:
histogram = go.Figure(
    go.Bar(
        x=h_grid[selected_grid_mask],
        y=selection_counts[selected_grid_mask],
        marker=dict(color="#ff7f0e"),
        hovertemplate=(
            "bandwidth h=%{x:.6g}<br>frequency=%{y}<extra></extra>"
        ),
        name="selected bandwidths",
    )
)

histogram.update_xaxes(
    title_text="selected bandwidth h",
    type="log",
)
histogram.update_yaxes(
    title_text="frequency among M repetitions",
    dtick=1,
)
histogram.update_layout(
    title=(
        f"Histogram of {M} Toeplitz-corrected bandwidth selections "
        f"(n={n}, α={alpha:g}, τ={tau:g})"
    ),
    width=950,
    height=520,
    template="plotly_white",
    bargap=0.08,
    showlegend=False,
)
histogram.show()

In [25]:
h_grid

array([0.02   , 0.02095, 0.02195, 0.023  , 0.02409, 0.02524, 0.02644,
       0.0277 , 0.02902, 0.0304 , 0.03185, 0.03336, 0.03495, 0.03661,
       0.03836, 0.04018, 0.0421 , 0.0441 , 0.0462 , 0.0484 , 0.05071,
       0.05312, 0.05565, 0.0583 , 0.06108, 0.06399, 0.06703, 0.07022,
       0.07357, 0.07707, 0.08074, 0.08458, 0.08861, 0.09283, 0.09725,
       0.10188, 0.10673, 0.11182, 0.11714, 0.12272, 0.12856, 0.13468,
       0.1411 , 0.14781, 0.15485, 0.16223, 0.16995, 0.17804, 0.18652,
       0.1954 , 0.20471, 0.21445, 0.22466, 0.23536, 0.24657, 0.25831,
       0.27061, 0.28349, 0.29699, 0.31114, 0.32595, 0.34147, 0.35773,
       0.37476, 0.39261, 0.4113 , 0.43089, 0.4514 , 0.4729 , 0.49542,
       0.519  , 0.54372, 0.56961, 0.59673, 0.62514, 0.65491, 0.68609,
       0.71876, 0.75299, 0.78884, 0.8264 , 0.86575, 0.90698, 0.95016,
       0.9954 , 1.0428 , 1.09246, 1.14447, 1.19897, 1.25606, 1.31587,
       1.37852, 1.44416, 1.51293, 1.58497, 1.66044, 1.7395 , 1.82233,
       1.9091 , 2.  

In [26]:
selected_indices

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0])